In [ ]:
"""
Initial imports and function declarations
"""

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from tqdm.notebook import tqdm, trange
import firedrake
from firedrake import (
    eq,
    conditional,
    min_value,
    max_value,
    exp,
    sqrt,
    inner,
    sym,
    tr,
    grad,
    Constant,
    dx,
    ds,
    dS,
    avg,
    jump,
)
import irksome
from irksome import Dt, lag
from icepack.constants import (
    weertman_sliding_law,
    glen_flow_law,
    gravity as g,
    ice_density as ρ_I,
    water_density as ρ_W,
)
import firedrake
import ufl

def get_test_function(q):
    z, = ufl.algorithms.extract_coefficients(q)
    w = firedrake.TestFunction(z.function_space())
    return firedrake.replace(q, {z: w})



n = 3 #Constant(glen_flow_law)
m = 3 #Constant(weertman_sliding_law)


def friction_law(**kwargs):
    τ, u, h = map(kwargs.get, ("basal_stress", "velocity", "thickness"))
    parameters = ("sliding_coefficient", "sliding_exponent")
    K, m = map(kwargs.get, parameters)

    p_W = lag(ρ_W * g * max_value(0, -(s - h)))
    p_I = ρ_I * g * h
    r = Constant(0.99)
    ϕ = lag(conditional(p_W > r * p_I, 0, 1))

    σ = get_test_function(τ)
    τ_2 = inner(τ, τ)

    if m == 1:
        τ_m = Constant(1.0)
    elif m == 3:
        τ_m = τ_2
    elif m == 5:
        τ_m = τ_2 ** 2
    else:
        raise ValueError("Sliding law exponent must be in [1, 3, 5]!")

    f = ϕ if m == 1 else ϕ ** m
    return inner(K * τ_m * τ + f * u, σ) * dx


def flow_law(**kwargs):
    variables = ("velocity", "membrane_stress", "thickness")
    u, M, h = map(kwargs.get, variables)
    parameters = ("flow_law_coefficient", "flow_law_exponent")
    A, n = map(kwargs.get, parameters)

    N = get_test_function(M)
    d = mesh.geometric_dimension
    M_2 = (inner(M, M) - tr(M)**2 / (d + 1)) / 2

    if n == 1:
        M_n = Constant(1.0)
    elif n == 3:
        M_n = M_2
    elif n == 5:
        M_n = M_2 ** 2
    else:
        raise ValueError("Flow law exponent must be in [1, 3, 5]!")

    ε = sym(grad(u))
    return h * (A * M_n * (inner(M, N) - tr(M) * tr(N) / (d + 1)) - inner(ε, N)) * dx


def momentum_balance(**kwargs):
    variables = (
        "velocity", "membrane_stress", "basal_stress", "thickness", "surface"
    )
    u, M, τ, h, s = map(kwargs.get, variables)
    v = get_test_function(u)

    ε = sym(grad(v))
    cell_balance = (-h * inner(M, ε) + inner(τ - ρ_I * g * h * grad(s), v)) * dx

    ν = firedrake.FacetNormal(mesh)
    facet_balance = ρ_I * g * avg(h) * inner(jump(s, ν), avg(v)) * dS

    return cell_balance + facet_balance


def terminus(**kwargs):
    variables = ("velocity", "thickness", "surface", "terminus_ids")
    u, h, s, terminus_ids = map(kwargs.get, variables)
    v = get_test_function(u)

    d = firedrake.min_value(s - h, 0)
    τ_I = ρ_I * g * h**2 / 2
    τ_W = ρ_W * g * d**2 / 2

    ν = firedrake.FacetNormal(mesh)
    return (τ_I - τ_W) * inner(v, ν) * ds(terminus_ids)


def smooth_max(a, b, ϵ):
    return (a + b + ((a - b)**2 + ϵ**2) ** 0.5) / 2



def mass_balance(**kwargs):
    variables = ("thickness", "velocity", "accumulation", "thickness_in")
    h, u, a, h_in = map(kwargs.get, variables)
    η = get_test_function(h)

    ν = firedrake.FacetNormal(mesh)
    F_cells = (Dt(h) * η - inner(h * u, grad(η)) - a * η) * dx
    f = h * max_value(0, inner(u, ν))
    F_facets = jump(f) * jump(η) * dS
    F_outflow = f * η * ds
    F_inflow = h_in * min_value(0, inner(u, ν)) * η * ds
    return F_cells + F_facets + F_outflow + F_inflow

In [ ]:
"""
Initialization of the bed geometry and velocity/height profiles
"""

import firedrake

lx = 610e3
nx = 450

mesh = firedrake.IntervalMesh(nx, lx, name="mesh")
thickness_element = firedrake.FiniteElement("DG", "interval", 0)
bed_element = firedrake.FiniteElement("CG", "interval", 1)
degree = 1

velocity_element = firedrake.FiniteElement("CG", "interval", degree)
stress_element = firedrake.FiniteElement("DG", "interval", degree - 1)

Q = firedrake.FunctionSpace(mesh, thickness_element)
S = firedrake.FunctionSpace(mesh, bed_element)
V = firedrake.VectorFunctionSpace(mesh, velocity_element)
Σ = firedrake.TensorFunctionSpace(mesh, stress_element, symmetry=True)

Lx = Constant(lx)

In [ ]:

"""
We utilize the mismip+ bed
"""
    
B_0 = Constant(-150)
B_2 = Constant(-728.8)
B_4 = Constant(343.91)
B_6 = Constant(-50.57)
x_c = Constant(300e3)


def mismip_bed(mesh):
    x, = firedrake.SpatialCoordinate(mesh)
    X = x / x_c
    B_x = B_0 + B_2 * X**2 + B_4 * X**4 + B_6 * X**6
    z_deep = Constant(-720)    
    return B_x


def linear_bed(mesh):
    x, = firedrake.SpatialCoordinate(mesh)
    b_in, b_out = 200, -400
    b_x = (b_in - (b_in - b_out) * x / Lx)
    return b_x

In [ ]:
x = firedrake.SpatialCoordinate(mesh)[0]

b = firedrake.Function(S).interpolate(mismip_bed(mesh))

h_in = Constant(200.0)
δh = Constant(100.0)
h_expr = h_in - δh * x / Lx
h_0 = firedrake.Function(Q).interpolate(h_expr)
h = h_0.copy(deepcopy=True)

ϵ = Constant(0.1)
s_expr = smooth_max(b + h, (1 - ρ_I / ρ_W) * h, ϵ)
s0 = firedrake.Function(Q).interpolate(s_expr)
z_b = firedrake.Function(Q).interpolate(s0 - h)

In [ ]:
fig, axes = plt.subplots()
firedrake.plot(b, axes=axes, edgecolor="tab:brown")
firedrake.plot(s0, axes=axes, edgecolor="tab:blue")
firedrake.plot(z_b, axes=axes, edgecolor="tab:blue")
axes.set_xlabel('Distance along flowline (m)')
axes.set_ylabel('Elevation')
plt.show()

In [ ]:
δu = firedrake.Constant(90.0)
expr = firedrake.as_vector([δu * x / Lx])  # 1-component vector
u_0 = firedrake.Function(V).interpolate(expr)

In [ ]:
A = Constant(20)
K = Constant(1e6)

τ_c = Constant(0.01)
ε_c = Constant(A * τ_c ** n)
u_c = Constant(K * τ_c ** m)

In [ ]:
print(f"Strain rate: {1000 * float(ε_c):0.3f} m / yr / km")
print(f"Speed:       {float(u_c):0.3f} m / yr")
print(f"at stress of {1000 * float(τ_c):0.3f} kPa")

In [ ]:
"""
Initialization of function spaces and mass/momentum balance
"""

Z = V * Σ * V
z = firedrake.Function(Z)
z.sub(0).assign(u_0);

inflow_ids = (1,)
terminus_ids = (2,)

#s = smooth_max(b + h, (1 - ρ_I / ρ_W) * h, ϵ)
s = max_value(b + h, (1 - ρ_I / ρ_W) * h)

u, M, τ = firedrake.split(z)
fields = {
    "velocity": u,
    "thickness": h,
    "surface": s,
    "membrane_stress": M,
    "basal_stress": τ,
}

parameters_1 = {
    "flow_law_coefficient": ε_c / τ_c,
    "flow_law_exponent": 1,
    "sliding_coefficient": u_c / τ_c,
    "sliding_exponent": 1,
}

parameters_3 = {
    "flow_law_coefficient": ε_c / τ_c ** n,
    "flow_law_exponent": n,
    "sliding_coefficient": u_c / τ_c ** m,
    "sliding_exponent": m,
}

α = Constant(1e-3)
F_flow_law_1 = flow_law(**fields, **parameters_1)
F_flow_law_3 = flow_law(**fields, **parameters_3)
F_flow_law = (1 / (1 + α) * F_flow_law_3 + α / (1 + α) * F_flow_law_1)

F_sliding_law_1 = friction_law(**fields, **parameters_1)
F_sliding_law_3 = friction_law(**fields, **parameters_3)
F_sliding_law = (1 / (1 + α) * F_sliding_law_3 + α / (1 + α) * F_sliding_law_1)

F_balance = momentum_balance(**fields)
F_terminus = terminus(**fields, terminus_ids=terminus_ids)

F = F_flow_law + F_sliding_law + F_balance + F_terminus

inflow_bc = firedrake.DirichletBC(Z.sub(0), u_0, inflow_ids)
bcs = [inflow_bc]

fparams = {"quadrature_degree": 8}
pparams = {"bcs": bcs, "form_compiler_parameters": fparams}
problem = firedrake.NonlinearVariationalProblem(F, z, **pparams)

logfile = ":mismipp-dual.log"
sparams = {
    "solver_parameters": {
        "snes_max_it": 200,
        "snes_linesearch_type": "nleqerr",
        "snes_monitor": logfile,
        "pc_factor_mat_solver_type": "umfpack",
    }
}
solver = firedrake.NonlinearVariationalSolver(problem, **sparams)
solver.solve()
u, M, τ = z.subfunctions

In [ ]:
fig, axes = plt.subplots()
firedrake.plot(firedrake.Function(S).interpolate(u[0]), axes=axes);

In [ ]:
W = V * Σ * V * Q

w = firedrake.Function(W)
w.sub(0).assign(z.sub(0))
w.sub(1).assign(z.sub(1))
w.sub(2).assign(z.sub(2))
w.sub(3).assign(h);

In [ ]:
inflow_bc = firedrake.DirichletBC(W.sub(0), u_0, inflow_ids)
bcs = [inflow_bc]

u, M, τ, h = firedrake.split(w)
v, N, σ, η = firedrake.TestFunctions(W)

s = lag(max_value(b + h, (1 - ρ_I / ρ_W) * h))
fields = {
    "velocity": u,
    "thickness": h,
    "surface": s,
    "membrane_stress": M,
    "basal_stress": τ,
}

H = Constant(500.0)

F_flow_law_1 = flow_law(**(fields | {"thickness": H}), **parameters_1)
F_flow_law_3 = flow_law(**(fields | {"thickness": H}), **parameters_3)
F_flow_law = (1 / (1 + α) * F_flow_law_3 + α / (1 + α) * F_flow_law_1)

F_sliding_law_1 = friction_law(**fields, **parameters_1)
F_sliding_law_3 = friction_law(**fields, **parameters_3)
F_sliding_law = (1 / (1 + α) * F_sliding_law_3 + α / (1 + α) * F_sliding_law_1)

F_balance = momentum_balance(**fields)
F_terminus = terminus(**fields, terminus_ids=terminus_ids)

F_momentum = F_flow_law + F_sliding_law + F_balance + F_terminus

In [ ]:
a_0 = Constant(1.0)
m_0 = Constant(1.125)
z_b = s - h
heaviside = lambda z: 1 / (1 + firedrake.exp(-z))
#a_expr = a_0 - m_0 * heaviside(-z_b) * heaviside(z_b - b)
a_expr = a_0 - m_0 * heaviside(z_b - b)

In [ ]:
F_mass = mass_balance(
    thickness=h, velocity=u, accumulation=lag(a_expr), thickness_in=h_0
)
F = F_momentum + F_mass

In [ ]:
"""
Main driver:
"""

method = irksome.BackwardEuler()
t = Constant(0.0)
timestep = 0.2
dt = Constant(timestep)

fparams = {"quadrature_degree": 8}
sparams = {
    "snes_monitor": ":mismip-dual-ts.log",
    "snes_type": "vinewtonrsls",
    "snes_linesearch_type": "bt", 
    "snes_linesearch_max_it": 40,
    "snes_atol": 2e-8,
    "pc_factor_mat_solver_type": "mumps",
}
lower = firedrake.Function(W)
upper = firedrake.Function(W)
lower.assign(-np.inf)
lower.sub(3).assign(0.0)
upper.assign(+np.inf)

params = {
    "bcs": bcs,
    "form_compiler_parameters": fparams,
    "solver_parameters": sparams,
    "stage_type": "value",
    "basis_type": "Bernstein",
    "bounds": ("stage", lower, upper)
}

solver = irksome.TimeStepper(F, method, t, dt, w, **params)

In [ ]:
ws = [w.copy(deepcopy=True)]
final_time = 3000.0
num_steps = int(final_time / timestep)

for step in trange(num_steps):
    solver.advance()
    ws.append(w.copy(deepcopy=True))

In [ ]:
u, M, τ, h = w.subfunctions

In [ ]:
s = firedrake.Function(Q).interpolate(max_value(b + h, (1 - ρ_I / ρ_W) * h))
z_b = firedrake.Function(Q).interpolate(s - h)

In [ ]:
fig, ax = plt.subplots()
ax.set_ylabel("Elevation (m)")
firedrake.plot(s, edgecolor="tab:blue", axes=ax)
firedrake.plot(z_b, edgecolor="tab:blue", axes=ax)
firedrake.plot(b, edgecolor="tab:brown", axes=ax)

ax = ax.twinx()
ax.set_ylabel("Speed (m/yr)", color="tab:orange")
ax.tick_params(axis="y", labelcolor="tab:orange")
u_x = firedrake.Function(S).interpolate(u[0])
firedrake.plot(u_x, edgecolor="tab:orange", axes=ax);